# SmartQ — Data Understanding & Exploratory Data Analysis (EDA)

This notebook covers the **Data Understanding** stage of the SmartQ CRISP-DM workflow.

In simple terms, **EDA is the main practical part of Data Understanding**, but Data Understanding is slightly broader. It also includes checking data quality, understanding the prediction target, identifying useful features, spotting missing values/outliers, and identifying fields that would cause data leakage.

**Prediction target:** `actual_wait_minutes`

**Dataset:** 100,000 synthetic SmartQ operational queue records.


## 1. Load the dataset

The code below works whether the notebook is run from the repository root or from the `notebooks/` folder.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATASET_NAME = "SmartQ_Synthetic_Operational_Dataset_100k.csv"

candidate_paths = [
    Path("data") / DATASET_NAME,
    Path("..") / "data" / DATASET_NAME,
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Run this notebook from the repository root "
        "or from the notebooks folder."
    )

df = pd.read_csv(data_path)

print("Dataset:", data_path.resolve())
print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
display(df.head())


## 2. Parse timestamps and define the modelling population

The waiting-time regression models will use **completed visits only**, because cancelled and no-show records do not contain a genuine completed waiting-time outcome.


In [ ]:
datetime_columns = [
    "scenario_date",
    "appointment_at",
    "arrival_at",
    "check_in_at",
    "service_eligible_at",
    "call_time",
    "service_started_at",
    "service_completed_at",
]

for column in datetime_columns:
    df[column] = pd.to_datetime(df[column], errors="coerce")

completed = df[df["status"] == "COMPLETED"].copy()

print(f"All records: {len(df):,}")
print(f"Completed records available for regression: {len(completed):,}")
print(f"Excluded no-shows/cancellations: {len(df) - len(completed):,}")


## 3. Basic structure and data types


In [ ]:
display(df.info())
display(df.describe(include="all").T)


## 4. Status, service, branch and queue composition


In [ ]:
for column in [
    "status",
    "branch_name",
    "service_name",
    "booking_source",
    "queue_type",
    "day_of_week",
    "is_peak_period",
]:
    print(f"\n--- {column} ---")
    display(df[column].value_counts(dropna=False).to_frame("count"))


## 5. Missing-value analysis

Some missing values are expected by design:

- `appointment_at` is blank for walk-ins.
- Recent-history features can be blank near the start of an operating day because there may not yet be enough earlier completed/called customers.
- Outcome fields are blank for no-shows and cancellations.


In [ ]:
missing_all = (
    df.isna()
      .sum()
      .to_frame("missing_rows")
      .assign(missing_pct=lambda x: (x["missing_rows"] / len(df) * 100).round(2))
      .query("missing_rows > 0")
      .sort_values("missing_rows", ascending=False)
)
display(missing_all)

print("\nMissing values inside completed visits only:")
missing_completed = (
    completed.isna()
             .sum()
             .to_frame("missing_rows")
             .assign(missing_pct=lambda x: (x["missing_rows"] / len(completed) * 100).round(2))
             .query("missing_rows > 0")
             .sort_values("missing_rows", ascending=False)
)
display(missing_completed)


## 6. Data-quality checks

These checks verify that the synthetic records obey basic SmartQ queue logic.


In [ ]:
appointment_rows = completed[completed["booking_source"] == "APPOINTMENT"]
walkin_rows = completed[completed["booking_source"] == "WALK_IN"]

quality_checks = pd.Series({
    "negative_wait_rows": int((completed["actual_wait_minutes"] < 0).sum()),
    "nonpositive_service_rows": int((completed["actual_service_minutes"] <= 0).sum()),
    "queue_position_mismatch": int(
        (completed["queue_position"] != completed["people_ahead"] + 1).sum()
    ),
    "call_before_service_eligibility": int(
        (completed["call_time"] < completed["service_eligible_at"]).sum()
    ),
    "service_start_before_call": int(
        (completed["service_started_at"] < completed["call_time"]).sum()
    ),
    "completion_before_service_start": int(
        (completed["service_completed_at"] < completed["service_started_at"]).sum()
    ),
    "appointment_eligible_before_booked_time": int(
        (appointment_rows["service_eligible_at"] < appointment_rows["appointment_at"]).sum()
    ),
    "walkin_eligibility_not_equal_checkin": int(
        (walkin_rows["service_eligible_at"] != walkin_rows["check_in_at"]).sum()
    ),
}, name="violations")

display(quality_checks.to_frame())


## 7. Understand the target: actual waiting time


In [ ]:
wait_stats = completed["actual_wait_minutes"].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
)
display(wait_stats.to_frame("actual_wait_minutes"))

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(completed["actual_wait_minutes"], bins=60)
ax.set_title("Distribution of Actual SmartQ Waiting Time")
ax.set_xlabel("Actual wait (minutes)")
ax.set_ylabel("Customers")
plt.show()


## 8. Waiting time by service, branch, queue lane and booking source


In [ ]:
group_columns = [
    "service_name",
    "branch_name",
    "queue_type",
    "booking_source",
    "is_peak_period",
]

for column in group_columns:
    summary = (
        completed.groupby(column)["actual_wait_minutes"]
                 .agg(["count", "mean", "median"])
                 .round(2)
                 .sort_values("mean", ascending=False)
    )
    print(f"\n--- Wait by {column} ---")
    display(summary)


In [ ]:
branch_wait = (
    completed.groupby("branch_name")["actual_wait_minutes"]
             .mean()
             .sort_values()
)

fig, ax = plt.subplots(figsize=(9, 5))
branch_wait.plot(kind="barh", ax=ax)
ax.set_title("Average Waiting Time by Branch")
ax.set_xlabel("Average wait (minutes)")
ax.set_ylabel("Branch")
plt.tight_layout()
plt.show()


## 9. Time-of-day and weekday patterns


In [ ]:
hourly_wait = completed.groupby("hour_of_day")["actual_wait_minutes"].mean()
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"]
weekday_wait = (
    completed.groupby("day_of_week")["actual_wait_minutes"]
             .mean()
             .reindex(weekday_order)
)

display(hourly_wait.round(2).to_frame("avg_wait_minutes"))
display(weekday_wait.round(2).to_frame("avg_wait_minutes"))

fig, ax = plt.subplots(figsize=(9, 5))
hourly_wait.plot(marker="o", ax=ax)
ax.set_title("Average Waiting Time by Hour")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Average wait (minutes)")
ax.grid(alpha=0.25)
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
weekday_wait.plot(kind="bar", ax=ax)
ax.set_title("Average Waiting Time by Weekday")
ax.set_xlabel("Day")
ax.set_ylabel("Average wait (minutes)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 10. Queue conditions versus waiting time

This section checks whether operational queue-state features behave in the direction we would expect before modelling.


In [ ]:
numeric_features = [
    "people_ahead",
    "general_waiting",
    "priority_waiting",
    "serving_count",
    "effective_open_counters",
    "counter_utilisation",
    "queue_pressure_index",
    "workload_minutes_ahead",
    "recent_avg_service_minutes_10",
    "recent_avg_wait_minutes_10",
    "recent_throughput_60m",
    "service_target_minutes",
    "hour_of_day",
    "actual_wait_minutes",
]

correlations = completed[numeric_features].corr(numeric_only=True)["actual_wait_minutes"]
correlations = correlations.drop("actual_wait_minutes").sort_values(ascending=False)

display(correlations.to_frame("correlation_with_actual_wait"))


In [ ]:
sample = completed.sample(n=min(5000, len(completed)), random_state=42)

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(
    sample["queue_pressure_index"],
    sample["actual_wait_minutes"],
    alpha=0.20,
    s=12,
)
ax.set_title("Queue Pressure vs Actual Waiting Time")
ax.set_xlabel("Queue pressure index")
ax.set_ylabel("Actual wait (minutes)")
plt.show()


## 11. Existing deterministic ETA as an engineering benchmark

This is **not one of the three required ML models**. It is useful as an extra comparison because SmartQ already has a deterministic ETA estimate.


In [ ]:
baseline_error = completed["actual_wait_minutes"] - completed["baseline_eta_minutes"]

baseline_mae = baseline_error.abs().mean()
baseline_rmse = np.sqrt((baseline_error ** 2).mean())

print(f"Deterministic SmartQ ETA MAE:  {baseline_mae:.2f} minutes")
print(f"Deterministic SmartQ ETA RMSE: {baseline_rmse:.2f} minutes")


## 12. Leakage audit

For predicting wait time at check-in, the following fields must not be used as model inputs because they contain information that becomes available only after the prediction point or directly contain the answer.


In [ ]:
target = "actual_wait_minutes"

leakage_columns = [
    "call_time",
    "actual_wait_minutes",
    "wait_variance_minutes",
    "actual_service_minutes",
    "service_variance_minutes",
    "counter_number",
    "service_started_at",
    "service_completed_at",
    "status",
    "no_show",
    "service_within_target",
    "wait_within_30_minutes",
]

display(pd.DataFrame({"excluded_leakage_or_outcome_field": leakage_columns}))


## 13. Initial EDA conclusions

After running the notebook, confirm the following before moving to modelling:

- Dataset dimensions and status counts are correct.
- Completed visits form the regression population.
- Expected missing values are understood rather than blindly deleted.
- No impossible negative waits or broken timestamp ordering exist.
- Waiting time varies with queue conditions, branch/time patterns and queue lane.
- The prediction target is right-skewed, so both MAE and RMSE are useful.
- Leakage fields are excluded from model inputs.
- The next stage is **Data Preparation**: choose features, handle missing history fields, encode categories and create a chronological train/validation/test split.
